# Predicting the Annual Turnover of a Restaurant  (CatBoost)
---

Goal:
- Train on Train_dataset_.csv
- Predict Annual Turnover for Test_dataset_.csv
- Submit CSV with:
    1) Registration Number
    2) Annual Turnover

Metric:
- RMSE (lower is better)

Why CatBoost?
- Handles categorical columns well (no heavy one-hot encoding needed)

Key trick:
- Turnover is usually skewed, so we train on log1p(target) and convert back with expm1.

---

## 2. Importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

In [ ]:
# 1) SETTINGS (edit only these if needed)
# =========================
TRAIN_PATH = "Train_dataset_.csv"
TEST_PATH  = "Test_dataset_.csv"

TARGET_COL = "Annual Turnover"
ID_COL     = "Registration Number"

N_SPLITS   = 3        # cross-validation folds
N_BINS     = 10       # bins for stratifying regression folds


In [35]:
# 2) LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

# Basic checks
for col in [TARGET_COL, ID_COL]:
    if col not in train_df.columns:
        raise ValueError(f"'{col}' not found in train dataset.")
if ID_COL not in test_df.columns:
    raise ValueError(f"'{ID_COL}' not found in test dataset.")

Train shape: (3493, 34)
Test shape : (500, 33)


In [36]:
# 3) SPLIT FEATURES / TARGET
# =========================
y = train_df[TARGET_COL].copy()
X = train_df.drop(columns=[TARGET_COL]).copy()

X_test = test_df.copy()

# IMPORTANT:
# Registration Number is an identifier, not a predictive feature.
# Keeping IDs as features can cause leakage / unstable generalisation.
X = X.drop(columns=[ID_COL])
X_test = X_test.drop(columns=[ID_COL])

# Align test columns to train columns (safe if any mismatch)
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)

assert list(X.columns) == list(X_test.columns), "Train/Test columns still do not match!"
print("✔ Train/Test columns aligned:", X.shape[1], "features")

✔ Train/Test columns aligned: 32 features


In [37]:
# 4) IDENTIFY CATEGORICAL COLS + SIMPLE MISSING VALUE HANDLING
# =========================
# Categorical columns: object / category
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Fill categorical missing -> "Unknown", cast to string (CatBoost-friendly)
for c in cat_cols:
    X[c] = X[c].fillna("Unknown").astype(str)
    X_test[c] = X_test[c].fillna("Unknown").astype(str)

# Fill numeric missing -> median (computed from train, applied to both)
for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

# CatBoost needs categorical indices
cat_features_idx = [X.columns.get_loc(c) for c in cat_cols]

print("Num features:", X.shape[1])
print("Categorical features:", len(cat_cols))

Num features: 32
Categorical features: 7


In [ ]:
# 5) CV FUNCTION (Stratified folds for regression)
# =========================
def make_bins(y_series: pd.Series, n_bins=8) -> pd.Series:
    return pd.qcut(y_series, q=n_bins, duplicates="drop").astype(str)

def cv_rmse_log_target(X, y, cat_features_idx, params, n_splits=3, n_bins=8, seed=42):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    y_bins = make_bins(y, n_bins=n_bins)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    rmses = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=seed,
            verbose=0
        )

        model.fit(
            X_tr, np.log1p(y_tr),
            cat_features=cat_features_idx,
            eval_set=(X_val, np.log1p(y_val)),
            use_best_model=True
        )

        pred = np.expm1(model.predict(X_val))
        pred = np.clip(pred, 0, None)

        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)
               
    return float(np.mean(rmses))

In [39]:
# 6) SIMPLE TUNING (depth -> l2 -> subsample)
# =========================
# Baseline training stability
BASE_PARAMS = dict(
    iterations=2500,          # FAST: smaller than 8000+
    learning_rate=0.06,       # FAST: learn quicker
    early_stopping_rounds=150 # FAST: stop sooner
)

def pick_best(grid, build_params, label):
    results = {}
    print(f"\n--- FAST TUNING: {label} ---")
    for v in grid:
        params = build_params(v)
        score = cv_rmse_log_target(X, y, cat_features_idx, params, n_splits=N_SPLITS, n_bins=N_BINS)
        results[v] = score
        print(f"{label}={v:<6}  CV_RMSE={score:,.4f}")
    best_v = min(results, key=results.get)
    print(f"Best {label}: {best_v}  (RMSE={results[best_v]:,.4f})")
    return best_v, results

# Tune depth (most impactful)
depth_grid = [6, 8, 10]
BEST_DEPTH, depth_results = pick_best(
    depth_grid,
    lambda d: {**BASE_PARAMS, "depth": d},
    "depth"
)

# Tune l2_leaf_reg (regularization)
l2_grid = [3, 5, 8]
BEST_L2, l2_results = pick_best(
    l2_grid,
    lambda l2: {**BASE_PARAMS, "depth": BEST_DEPTH, "l2_leaf_reg": l2},
    "l2_leaf_reg"
)

# Tune subsample (variance reduction)
sub_grid = [0.8, 1.0]
BEST_SUB, sub_results = pick_best(
    sub_grid,
    lambda s: {**BASE_PARAMS, "depth": BEST_DEPTH, "l2_leaf_reg": BEST_L2, "subsample": s},
    "subsample"
)

BEST_PARAMS = {**BASE_PARAMS, "depth": BEST_DEPTH, "l2_leaf_reg": BEST_L2, "subsample": BEST_SUB}
print("\n Best params:", BEST_PARAMS)




--- FAST TUNING: depth ---
depth=6       CV_RMSE=20,718,879.7491
depth=8       CV_RMSE=20,682,048.8433
depth=10      CV_RMSE=20,746,363.9594
Best depth: 8  (RMSE=20,682,048.8433)

--- FAST TUNING: l2_leaf_reg ---
l2_leaf_reg=3       CV_RMSE=20,682,048.8433
l2_leaf_reg=5       CV_RMSE=20,751,411.9734
l2_leaf_reg=8       CV_RMSE=20,703,330.1488
Best l2_leaf_reg: 3  (RMSE=20,682,048.8433)

--- FAST TUNING: subsample ---
subsample=0.8     CV_RMSE=20,682,048.8433
subsample=1.0     CV_RMSE=20,731,913.4200
Best subsample: 0.8  (RMSE=20,682,048.8433)

✅ Best params: {'iterations': 2500, 'learning_rate': 0.06, 'early_stopping_rounds': 150, 'depth': 8, 'l2_leaf_reg': 3, 'subsample': 0.8}
BASE_PARAMS: {'iterations': 2500, 'learning_rate': 0.06, 'early_stopping_rounds': 150}


In [40]:
# 7) TRAIN FINAL MODEL ON FULL TRAIN SET (log-target)
# =========================
final_model = CatBoostRegressor(
    **BEST_PARAMS,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

final_model.fit(X, np.log1p(y), cat_features=cat_features_idx)


0:	learn: 0.5412443	total: 199ms	remaining: 8m 16s
200:	learn: 0.3788736	total: 27.5s	remaining: 5m 14s
400:	learn: 0.3159105	total: 56.3s	remaining: 4m 54s
600:	learn: 0.2645239	total: 1m 34s	remaining: 5m
800:	learn: 0.2240126	total: 2m 5s	remaining: 4m 26s
1000:	learn: 0.1906494	total: 2m 32s	remaining: 3m 48s
1200:	learn: 0.1636188	total: 3m 4s	remaining: 3m 19s
1400:	learn: 0.1416048	total: 3m 31s	remaining: 2m 45s
1600:	learn: 0.1219467	total: 3m 54s	remaining: 2m 11s
1800:	learn: 0.1047738	total: 4m 20s	remaining: 1m 41s
2000:	learn: 0.0918132	total: 4m 45s	remaining: 1m 11s
2200:	learn: 0.0801506	total: 5m 14s	remaining: 42.8s
2400:	learn: 0.0700078	total: 5m 41s	remaining: 14.1s
2499:	learn: 0.0652717	total: 5m 55s	remaining: 0us


In [41]:
# 8) PREDICT TEST + CREATE SUBMISSION
# =========================
test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    TARGET_COL: test_pred
})

submission.to_csv("submission_catboost.csv", index=False)
print("\n✔ Saved: submission_catboost.csv")
print(submission.head())


✔ Saved: submission_catboost.csv
   Registration Number  Annual Turnover
0                20001     2.821286e+07
1                20002     4.985074e+07
2                20003     3.250132e+07
3                20004     4.269235e+07
4                20005     4.350779e+07
